# 01 · Discovery notebook — all three Bronze sources

**Backend:** MinIO object storage
**Goal:** understand the raw data before building Silver — the recommended first
working session from `assignment/getting-started.md` (steps 2–7).

Sections:
1. Bronze object and archive-member inventory
2. Validate one record of each type on tiny samples
3. Schema and type profile per source
4. Missingness, duplicates, ranges, structural invariants
5. Candidate entities and keys
6. Evidence for rejected cross-source joins
7. One record, end to end: `bronze` → `silver` → `gold` → `ml`

Decisions and open questions go in `decision_log.md`, not here. This notebook
is exploration/profiling only — production transforms belong in
`src/quantum_lake_student/stages/`.

**Before this notebook:** run `make bootstrap`, then in a JupyterLab terminal
run `make check` and `make test` (step 1 of the recommended session — not
something a notebook can do for you).

**Before committing:** clear large cell outputs per `starter/notebooks/README.md`.
The outputs below are reference values computed against this release's exact
Bronze bytes — re-run each cell yourself to confirm your platform reproduces them,
then clear outputs before you commit.

## 0 · Setup and shared helpers

In [1]:
import io
import csv
import ast
import re
import zipfile

import yaml

from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client, bronze_inventory
from quantum_lake_student import formats

settings = Settings.from_environment()
client = minio_client(settings)

SYNDROMES_KEY = "bronze/source=qec_syndromes/syndromes_dataset.zip"
GOOGLE_KEY = "bronze/source=google_qec/google-surface-code-curated.zip"
QASMBENCH_KEY = "bronze/source=qasmbench/qasmbench-qec.zip"


Download each Bronze zip once and cache the bytes, so repeated reads across
cells don't re-download from MinIO.

In [2]:
_zip_cache: dict[str, bytes] = {}

def get_zip(bronze_key: str) -> zipfile.ZipFile:
    """Return a ZipFile for this Bronze object, downloading it once and caching the bytes."""
    if bronze_key not in _zip_cache:
        response = client.get_object(settings.s3_bucket, bronze_key)
        try:
            _zip_cache[bronze_key] = response.read()
        finally:
            response.close()
            response.release_conn()
    return zipfile.ZipFile(io.BytesIO(_zip_cache[bronze_key]))

def list_zip_members(bronze_key: str) -> list[tuple[str, int]]:
    with get_zip(bronze_key) as zf:
        return [(info.filename, info.file_size) for info in zf.infolist()]

def read_zip_member(bronze_key: str, member_name: str) -> bytes:
    with get_zip(bronze_key) as zf:
        return zf.read(member_name)


## 1 · Bronze object and archive-member inventory

`getting-started.md` states Bronze holds exactly three objects, one archive per
source. Confirm that, then list each archive's members without extracting to disk.

In [3]:
objects = bronze_inventory(settings)
for name, size in objects:
    print(f"{name:60s} {size:>12,d} bytes")
print(f"\ntotal bronze objects: {len(objects)}")


bronze/source=google_qec/google-surface-code-curated.zip       14,638,673 bytes
bronze/source=qasmbench/qasmbench-qec.zip                         144,172 bytes
bronze/source=qec_syndromes/syndromes_dataset.zip                 358,017 bytes

total bronze objects: 3


In [4]:
for key in (SYNDROMES_KEY, GOOGLE_KEY, QASMBENCH_KEY):
    members = list_zip_members(key)
    print(f"{key}  ->  {len(members)} member(s)")
    for filename, size in members[:6]:
        print(f"   {filename:70s} {size:>10,d} bytes")
    if len(members) > 6:
        print(f"   ... ({len(members) - 6} more)")
    print()


bronze/source=qec_syndromes/syndromes_dataset.zip  ->  8 member(s)
   d-3_pfr-0.000010_nb-10M.csv                                                 4,411 bytes
   d-3_pfr-0.000050_nb-10M.csv                                                13,715 bytes
   d-3_pfr-0.000100_nb-10M.csv                                                31,115 bytes
   d-3_pfr-0.000500_nb-10M.csv                                                89,438 bytes
   d-3_pfr-0.001000_nb-10M.csv                                               181,222 bytes
   d-3_pfr-0.005000_nb-10M.csv                                             1,323,765 bytes
   ... (2 more)

bronze/source=google_qec/google-surface-code-curated.zip  ->  76 member(s)
   README.txt                                                                 15,656 bytes
   surface_code_bX_d3_r25_center_3_5/circuit_detector_error_model.dem        172,277 bytes
   surface_code_bX_d3_r25_center_3_5/circuit_ideal.stim                       18,773 bytes
   surface_code_bX_d3_

### 1a · Read the syndromes README — check it against the actual header

In [5]:
print(read_zip_member(SYNDROMES_KEY, "README.txt").decode())


The file names: d-<surface_code_distance>_pfr-<physical_fault_rate>_nb-<number_of_samples>

The file format is a csv file with the following columns:
- label: binary label (0: no error, 1: error)
- syndromes: syndrome measurement sequence (tuples of the form (round, syndromes))
- quantity: number of samples for this label + syndrome sequence


In [6]:
with get_zip(SYNDROMES_KEY) as zf:
    with zf.open("d-3_pfr-0.000010_nb-10M.csv") as f:
        header = next(csv.reader(io.TextIOWrapper(f)))
print("Actual header:", header)


Actual header: ['labels', 'syndromes', 'quantity']


**Finding:** the README documents the column as `label` (singular), but the
actual CSV header is `labels` (plural). Logged in `decision_log.md` — harmless
here since we read by position/DictReader key from the real header, not from
the README, but worth flagging in case a later release "fixes" the header to
match the README and silently breaks a hardcoded column name.

## 2 · Validate one record of each type (tiny samples)

Before writing any bulk processing, decode **one** record from each source by
hand and check it against the documentation.

### 2a · One `qec_syndromes` CSV row

Use `csv.reader`, not a naive `line.split(",")` — the `syndromes` field contains
internal commas (it's a nested tuple rendered as text), so a naive split would
cut a single field into pieces.

In [7]:
with get_zip(SYNDROMES_KEY) as zf:
    with zf.open("d-3_pfr-0.000010_nb-10M.csv") as f:
        reader = csv.reader(io.TextIOWrapper(f))
        header = next(reader)
        rows = list(reader)

row = rows[1]
print("Header:", header)
print("Row 1:", row)

parsed = ast.literal_eval(row[1])
print("Parsed type:", type(parsed))
print("Rounds:", len(parsed), " Checks per round:", len(parsed[0]))
print("Values:", parsed)

flat16 = [bit for round_ in parsed for bit in round_]
print("Flattened 16 bits, round-first then check:", flat16)
assert len(flat16) == 16 and set(flat16) <= {0, 1}


Header: ['labels', 'syndromes', 'quantity']
Row 1: ['0', '((0, 0, 1, 0), (0, 0, 1, 0), (0, 0, 0, 0), (0, 0, 0, 0))', '486']
Parsed type: <class 'tuple'>
Rounds: 4  Checks per round: 4
Values: ((0, 0, 1, 0), (0, 0, 1, 0), (0, 0, 0, 0), (0, 0, 0, 0))
Flattened 16 bits, round-first then check: [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]


`labels` and `quantity` come back as strings from `csv.reader` — a cleaning
note for Silver (cast `labels` to `bool`, `quantity` to `int64`, both checked > 0
where relevant).

### 2b · One Google `b8` record (`measurements.b8`) and its `properties.yml`

In [8]:
exp_dir = "surface_code_bX_d3_r25_center_3_5"
with get_zip(GOOGLE_KEY) as zf:
    props = yaml.safe_load(zf.read(f"{exp_dir}/properties.yml"))
    meas_bytes = zf.read(f"{exp_dir}/measurements.b8")
    det_bytes = zf.read(f"{exp_dir}/detection_events.b8")

print(props)

meas_bits, det_bits = props["circuit_measurements"], props["circuit_detectors"]
rec_bytes = formats.b8_record_bytes(meas_bits)
print("bytes needed per measurement record:", rec_bytes, "(", meas_bits, "bits, byte-aligned)")

first_record = next(formats.iter_b8_records(meas_bytes, bits_per_record=meas_bits))
print("first measurement record, first 20 of", len(first_record), "bits:", first_record[:20])

first_det_record = next(formats.iter_b8_records(det_bytes, bits_per_record=det_bits))
print("first detector record, first 20 of", len(first_det_record), "bits:", first_det_record[:20])
print("fired detectors in shot 0:", sum(first_det_record))


{'type': 'surface_code_memory_experiment', 'basis': 'X', 'rounds': 25, 'distance': 3, 'data_qubits': 9, 'measure_qubits': 8, 'shots': 50000, 'center_data_qubit_row': 3, 'center_data_qubit_col': 5, 'circuit_measurements': 209, 'circuit_sweep_bits': 9, 'circuit_detectors': 200, 'circuit_observables': 1, 'circuit_qubits': 17}
bytes needed per measurement record: 27 ( 209 bits, byte-aligned)
first measurement record, first 20 of 209 bits: (1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1)
first detector record, first 20 of 200 bits: (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0)
fired detectors in shot 0: 27


209 measurement bits pad to 27 bytes (216 bits, 7 padding bits) — confirms the
byte-alignment/padding rule from `data-sources.md`.

### 2c · One Google `01` value (`obs_flips_actual.01`)

In [9]:
with get_zip(GOOGLE_KEY) as zf:
    actual_bytes = zf.read(f"{exp_dir}/obs_flips_actual.01")

actual = formats.parse_01_records(actual_bytes)
print("total lines:", len(actual), "== shots?", len(actual) == props["shots"])
print("first value (shot 0 actual_observable_flip):", actual[0])
print("value domain:", set(actual))


total lines: 50000 == shots? True
first value (shot 0 actual_observable_flip): 1
value domain: {0, 1}


### 2d · One QASMBench circuit — `qec_sm_n5.qasm` (the clearest parity-check example per `data-sources.md`)

In [10]:
with get_zip(QASMBENCH_KEY) as zf:
    qasm_text = zf.read("small/qec_sm_n5/qec_sm_n5.qasm").decode("utf-8")
print(qasm_text)


// Repetition code syndrome measurement
OPENQASM 2.0;
include "qelib1.inc";
qreg q[3];
qreg a[2];
creg c[3];
creg syn[2];
gate syndrome d1,d2,d3,a1,a2 
{ 
  cx d1,a1; cx d2,a1; 
  cx d2,a2; cx d3,a2; 
}
x q[0]; // error
barrier q;
syndrome q[0],q[1],q[2],a[0],a[1];
measure a -> syn;
if(syn==1) x q[0];
if(syn==2) x q[2];
if(syn==3) x q[1];
measure q -> c;



Two quantum registers (`q[3]` data, `a[2]` ancilla) — register-local index 0 in
`q` is **not** the same qubit as index 0 in `a`. The custom `gate syndrome ...`
body defines two parity checks (`q0⊕q1 -> a0`, `q1⊕q2 -> a1`); its `cx`
statements are a *definition*, not executed operations by themselves — only
the call `syndrome q[0],q[1],q[2],a[0],a[1];` executes it (4 `cx`, expanded).
Three `if(syn==...)` statements are the conditional corrections.

## 3 · Schema and type profile per source

Sizes, record counts, columns/dtypes for **all** files, not just the one sample above.

### 3a · `qec_syndromes` — all 7 CSVs

In [11]:
syn_summary = []
with get_zip(SYNDROMES_KEY) as zf:
    csv_members = sorted(m for m in zf.namelist() if m.endswith(".csv"))
    for filename in csv_members:
        with zf.open(filename) as f:
            reader = csv.reader(io.TextIOWrapper(f))
            next(reader)  # header
            rows = list(reader)

        pfr = float(re.search(r"pfr-([0-9.]+)", filename).group(1))
        total_quantity = sum(int(r[2]) for r in rows)
        labels_seen = {r[0] for r in rows}
        shapes = {(len(p := ast.literal_eval(r[1])), tuple(len(rnd) for rnd in p)) for r in rows}
        dup_keys = len(rows) - len({(r[0], r[1]) for r in rows})

        syn_summary.append(dict(file=filename, rows=len(rows), pfr=pfr,
                                 total_quantity=total_quantity, labels_seen=labels_seen,
                                 shapes=shapes, dup_keys=dup_keys))

for s in syn_summary:
    print(f"{s['file']:32s} rows={s['rows']:6d}  pfr={s['pfr']:.6f}  "
          f"sum(quantity)={s['total_quantity']:>10,d}  labels={s['labels_seen']}  "
          f"shapes={s['shapes']}  dup(labels,syndromes)={s['dup_keys']}")

grand_rows = sum(s["rows"] for s in syn_summary)
grand_qty = sum(s["total_quantity"] for s in syn_summary)
print(f"\nTOTAL rows (aggregate examples): {grand_rows:,}")
print(f"TOTAL quantity (weighted shots): {grand_qty:,}")


d-3_pfr-0.000010_nb-10M.csv      rows=    68  pfr=0.000010  sum(quantity)=10,000,000  labels={'1', '0'}  shapes={(4, (4, 4, 4, 4))}  dup(labels,syndromes)=0
d-3_pfr-0.000050_nb-10M.csv      rows=   215  pfr=0.000050  sum(quantity)=10,000,000  labels={'1', '0'}  shapes={(4, (4, 4, 4, 4))}  dup(labels,syndromes)=0
d-3_pfr-0.000100_nb-10M.csv      rows=   491  pfr=0.000100  sum(quantity)=10,000,000  labels={'1', '0'}  shapes={(4, (4, 4, 4, 4))}  dup(labels,syndromes)=0
d-3_pfr-0.000500_nb-10M.csv      rows=  1407  pfr=0.000500  sum(quantity)=10,000,000  labels={'1', '0'}  shapes={(4, (4, 4, 4, 4))}  dup(labels,syndromes)=0
d-3_pfr-0.001000_nb-10M.csv      rows=  2854  pfr=0.001000  sum(quantity)=10,000,000  labels={'1', '0'}  shapes={(4, (4, 4, 4, 4))}  dup(labels,syndromes)=0
d-3_pfr-0.005000_nb-10M.csv      rows= 20887  pfr=0.005000  sum(quantity)=10,000,000  labels={'1', '0'}  shapes={(4, (4, 4, 4, 4))}  dup(labels,syndromes)=0
d-3_pfr-0.010000_nb-10M.csv      rows= 49676  pfr=0.010000

Matches the brief exactly: **75,598 aggregate rows** representing **70,000,000**
weighted shots, 10M per file as the filename promises, all `4×4` shaped, zero
`(labels, syndromes)` duplicate pairs within any file.

### 3b · `google_qec` — all 5 experiments

In [12]:
google_summary = []
with get_zip(GOOGLE_KEY) as zf:
    exp_dirs = sorted({m.split("/")[0] for m in zf.namelist() if "/" in m})
    for d in exp_dirs:
        props = yaml.safe_load(zf.read(f"{d}/properties.yml"))
        shots = props["shots"]
        meas_bits, det_bits, sweep_bits = (
            props["circuit_measurements"], props["circuit_detectors"], props["circuit_sweep_bits"]
        )
        sizes = {name: len(zf.read(f"{d}/{name}"))
                  for name in ("measurements.b8", "detection_events.b8", "sweep.b8")}
        expected = {
            "measurements.b8": shots * formats.b8_record_bytes(meas_bits),
            "detection_events.b8": shots * formats.b8_record_bytes(det_bits),
            "sweep.b8": shots * formats.b8_record_bytes(sweep_bits),
        }
        actual_lines = len(formats.parse_01_records(zf.read(f"{d}/obs_flips_actual.01")))
        decoder_files = sorted(n.split("/")[-1] for n in zf.namelist()
                                if n.startswith(f"{d}/obs_flips_predicted_by_"))
        google_summary.append(dict(dir=d, distance=props["distance"], basis=props["basis"],
                                    rounds=props["rounds"], shots=shots,
                                    sizes_match={k: sizes[k] == expected[k] for k in sizes},
                                    actual_lines=actual_lines, decoders=len(decoder_files)))

for s in google_summary:
    print(f"{s['dir']:38s} d={s['distance']} basis={s['basis']} rounds={s['rounds']:2d} "
          f"shots={s['shots']:6,d} b8_len_ok={all(s['sizes_match'].values())} "
          f"actual.01_lines={s['actual_lines']:6,d} decoders={s['decoders']}")

print(f"\nTOTAL shots across all experiments: {sum(s['shots'] for s in google_summary):,}")


surface_code_bX_d3_r25_center_3_5      d=3 basis=X rounds=25 shots=50,000 b8_len_ok=True actual.01_lines=50,000 decoders=4
surface_code_bX_d3_r25_center_5_3      d=3 basis=X rounds=25 shots=50,000 b8_len_ok=True actual.01_lines=50,000 decoders=4
surface_code_bX_d3_r25_center_5_7      d=3 basis=X rounds=25 shots=50,000 b8_len_ok=True actual.01_lines=50,000 decoders=4
surface_code_bX_d3_r25_center_7_5      d=3 basis=X rounds=25 shots=50,000 b8_len_ok=True actual.01_lines=50,000 decoders=4
surface_code_bX_d5_r25_center_5_5      d=5 basis=X rounds=25 shots=50,000 b8_len_ok=True actual.01_lines=50,000 decoders=4

TOTAL shots across all experiments: 250,000


Matches the brief: **4 distance-3 + 1 distance-5 experiments, 250,000 shots total**, all companion `.b8` files byte-align exactly, every `obs_flips_actual.01` has one line per shot with all 4 decoder files present.

### 3c · `qasmbench` — all 3 circuit families × 2 variants

In [13]:
qasm_summary = []
with get_zip(QASMBENCH_KEY) as zf:
    qasm_members = sorted(m for m in zf.namelist() if m.endswith(".qasm"))
    for member in qasm_members:
        text = zf.read(member).decode("utf-8")
        qreg = re.findall(r"qreg (\w+)\[(\d+)\];", text)
        creg = re.findall(r"creg (\w+)\[(\d+)\];", text)
        qasm_summary.append(dict(
            member=member, variant="transpiled" if "transpiled" in member else "source",
            qreg=qreg, creg=creg, cx_count=len(re.findall(r"\bcx\b", text)),
            measure_count=len(re.findall(r"\bmeasure\b", text)),
            if_count=len(re.findall(r"\bif\s*\(", text)),
            custom_gates=re.findall(r"\bgate\s+(\w+)", text),
        ))

for s in qasm_summary:
    print(f"{s['member']:55s} qreg={s['qreg']} creg={s['creg']} "
          f"cx={s['cx_count']:2d} measure={s['measure_count']} if={s['if_count']} "
          f"custom_gates={s['custom_gates']}")


small/error_correctiond3_n5/error_correctiond3_n5.qasm  qreg=[('q', '5')] creg=[('c', '5')] cx=49 measure=5 if=0 custom_gates=[]
small/error_correctiond3_n5/error_correctiond3_n5_transpiled.qasm qreg=[('q', '5')] creg=[('c', '5')] cx=49 measure=5 if=0 custom_gates=[]
small/qec_en_n5/qec_en_n5.qasm                          qreg=[('q', '5')] creg=[('c', '5')] cx=10 measure=5 if=0 custom_gates=[]
small/qec_en_n5/qec_en_n5_transpiled.qasm               qreg=[('q', '5')] creg=[('c', '5')] cx=10 measure=5 if=0 custom_gates=[]
small/qec_sm_n5/qec_sm_n5.qasm                          qreg=[('q', '3'), ('a', '2')] creg=[('c', '3'), ('syn', '2')] cx= 4 measure=2 if=3 custom_gates=['syndrome']
small/qec_sm_n5/qec_sm_n5_transpiled.qasm               qreg=[('q', '3'), ('a', '2')] creg=[('c', '3'), ('syn', '2')] cx= 4 measure=5 if=3 custom_gates=[]


Only **`qec_sm_n5`** has explicit conditional corrections and a custom `gate syndrome` definition with separate data/ancilla registers — build the `stabilizer_check` / `conditional_correction` parsers against it first.

## 4 · Missingness, duplicates, ranges, structural invariants

### 4a · `qec_syndromes` — shape, binary domain, and missing values, every row of every file

In [14]:
issues = []
with get_zip(SYNDROMES_KEY) as zf:
    csv_members = sorted(m for m in zf.namelist() if m.endswith(".csv"))
    for filename in csv_members:
        with zf.open(filename) as f:
            reader = csv.reader(io.TextIOWrapper(f))
            next(reader)
            for i, row in enumerate(reader):
                if any(v.strip() == "" for v in row):
                    issues.append(f"{filename} row {i}: missing/empty value")
                    continue
                parsed = ast.literal_eval(row[1])
                if len(parsed) != 4 or any(len(r) != 4 for r in parsed):
                    issues.append(f"{filename} row {i}: wrong shape {parsed}")
                if any(v not in (0, 1) for r in parsed for v in r):
                    issues.append(f"{filename} row {i}: non-binary value {parsed}")

if issues:
    for issue in issues:
        print("ISSUE:", issue)
else:
    print("All 75,598 rows across all 7 files: correct 4x4 binary shape, no missing/empty fields.")


All 75,598 rows across all 7 files: correct 4x4 binary shape, no missing/empty fields.


### 4b · Duplicate `(labels, syndromes)` pairs and same-syndrome-both-labels, per experiment file

Scoped **per file** (per fault-rate experiment), not pooled across all seven —
files are separate simulation runs, so "is this a duplicate?" and "does this
syndrome occur under both labels?" are both questions about one experiment at
a time, not about the release as a whole.

In [15]:
with get_zip(SYNDROMES_KEY) as zf:
    csv_members = sorted(m for m in zf.namelist() if m.endswith(".csv"))
    for filename in csv_members:
        with zf.open(filename) as f:
            reader = csv.reader(io.TextIOWrapper(f))
            next(reader)
            seen = set()
            dup_count = 0
            label0, label1 = set(), set()
            for row in reader:
                key = (row[0], row[1])
                if key in seen:
                    dup_count += 1
                seen.add(key)
                (label0 if row[0] == "0" else label1).add(row[1])
        overlap = label0 & label1
        print(f"{filename}: duplicate (labels,syndromes) pairs={dup_count}  "
              f"syndromes seen under both labels={len(overlap)}")


d-3_pfr-0.000010_nb-10M.csv: duplicate (labels,syndromes) pairs=0  syndromes seen under both labels=0
d-3_pfr-0.000050_nb-10M.csv: duplicate (labels,syndromes) pairs=0  syndromes seen under both labels=13
d-3_pfr-0.000100_nb-10M.csv: duplicate (labels,syndromes) pairs=0  syndromes seen under both labels=46
d-3_pfr-0.000500_nb-10M.csv: duplicate (labels,syndromes) pairs=0  syndromes seen under both labels=179
d-3_pfr-0.001000_nb-10M.csv: duplicate (labels,syndromes) pairs=0  syndromes seen under both labels=467
d-3_pfr-0.005000_nb-10M.csv: duplicate (labels,syndromes) pairs=0  syndromes seen under both labels=6244
d-3_pfr-0.010000_nb-10M.csv: duplicate (labels,syndromes) pairs=0  syndromes seen under both labels=18210


**Two findings worth keeping:**

1. `(labels, syndromes)` is a duplicate-free key **within every file** — no file
   needs a surrogate id purely to deduplicate.
2. The both-labels overlap **grows sharply with physical fault rate**: 0 at
   `pfr=0.00001`, up to 18,210 patterns at `pfr=0.01` (out of 49,676 rows in
   that file — over a third). At low error rates, a syndrome pattern almost
   always implies one label; at high error rates it frequently doesn't. That's
   a genuinely useful thing to say in the design report about why
   `syndrome_bits` alone is a weak feature at high fault rates, and it's new —
   the earlier discovery pass only reported a single pooled-across-files count.

### 4c · `google_qec` — spot-check detector counts and decoder-prediction domains

In [16]:
with get_zip(GOOGLE_KEY) as zf:
    for d in exp_dirs[:1]:  # spot-check; full validation is pipeline work, not discovery
        props = yaml.safe_load(zf.read(f"{d}/properties.yml"))
        det_bits = props["circuit_detectors"]
        det_records = list(formats.iter_b8_records(zf.read(f"{d}/detection_events.b8"), bits_per_record=det_bits))
        counts = [sum(r) for r in det_records]
        print(d)
        print("  detector_event_count range:", min(counts), "-", max(counts), "of", det_bits, "possible")
        for decoder in ("belief_matching", "correlated_matching", "pymatching", "tensor_network_contraction"):
            vals = set(formats.parse_01_records(zf.read(f"{d}/obs_flips_predicted_by_{decoder}.01")))
            print(f"  {decoder} prediction domain: {vals}")


surface_code_bX_d3_r25_center_3_5
  detector_event_count range: 2 - 76 of 200 possible
  belief_matching prediction domain: {0, 1}
  correlated_matching prediction domain: {0, 1}
  pymatching prediction domain: {0, 1}
  tensor_network_contraction prediction domain: {0, 1}


## 5 · Candidate entities and keys

| Source | Candidate entity | Key | Cardinality / notes |
| --- | --- | --- | --- |
| `qec_syndromes` | aggregate syndrome observation | `(experiment_id, syndrome_bits, logical_error_label)` — confirmed duplicate-free within every file (§4b) | belongs to exactly one `physical_fault_rate` group; `quantity` is a weight, not a row multiplier; the same `syndrome_bits` legitimately repeats under both labels, more often as fault rate rises (§4b) |
| `google_qec` | experiment | `experiment_id` from directory name (`basis`+`distance`+`rounds`+`center_row`+`center_col`) | 1 experiment → many shots (1:N) |
| `google_qec` | shot | `(experiment_id, shot_index)` | 1 shot → 1 measurement record, 1 sweep record, 1 detector record, 1 actual label, 4 predictions, all aligned by row position across companion files (validated in §3b) |
| `qasmbench` | circuit | `(benchmark_name, variant)`, e.g. `(qec_sm_n5, source)` | 1 circuit → many stabilizer checks (1:N), 1 circuit → many conditional corrections (1:N, possibly zero — §3c) |
| `qasmbench` | stabilizer check | `(circuit_id, check_id)` | belongs to exactly one circuit |
| `qasmbench` | conditional correction | `(circuit_id, condition_register, condition_value)` if unique, else add statement order | belongs to exactly one circuit |

**Natural vs. surrogate key for `syndrome_observation`:** `(experiment_id,
syndrome_bits, logical_error_label)` is empirically unique in this release
(§4b — 0 duplicates in every file), so it's a valid natural key *today*. It's
still worth generating a `source_record_id` hash for `results/part1/
source_trace.parquet` regardless, per the shared tracing rule in
`silver-tables.md` — that's about traceability to Bronze, not about
uniqueness, so both can be true at once.

## 6 · Evidence for rejected cross-source joins

`data-sources.md` states QASMBench "does not identify the circuits used by the
Google experiments" and must not be joined row-by-row to Google or syndromes.
Concrete evidence to keep for the design report:

In [17]:
with get_zip(GOOGLE_KEY) as zf:
    google_names = sorted({m.split("/")[0] for m in zf.namelist() if "/" in m})
with get_zip(QASMBENCH_KEY) as zf:
    qasm_names = sorted({m.split("/")[1] for m in zf.namelist() if m.startswith("small/")})

print("Google experiment directory names:", google_names)
print("QASMBench benchmark names:", qasm_names)
print("Overlap:", set(google_names) & set(qasm_names))
print()
print("QASMBench circuits carry no 'distance', 'shots', or any other field that")
print("could match a Google experiment or a syndrome-file physical fault rate.")
print("No shared identifier exists in either direction.")


Google experiment directory names: ['surface_code_bX_d3_r25_center_3_5', 'surface_code_bX_d3_r25_center_5_3', 'surface_code_bX_d3_r25_center_5_7', 'surface_code_bX_d3_r25_center_7_5', 'surface_code_bX_d5_r25_center_5_5']
QASMBench benchmark names: ['error_correctiond3_n5', 'qec_en_n5', 'qec_sm_n5']
Overlap: set()

QASMBench circuits carry no 'distance', 'shots', or any other field that
could match a Google experiment or a syndrome-file physical fault rate.
No shared identifier exists in either direction.


Zero name overlap, no shared key field even conceptually. All three sources are
stored in the same Gold database but kept as separate entities; no cross-source
join is attempted, and this cell is the evidence `ASSIGNMENT_SPEC.md` §5 asks
teams to preserve ("Teams must preserve the evidence that no row-level
QASMBench-to-experiment join exists.").

## 7 · One record, end to end: `bronze` → `silver` → `gold` → `ml`

Not the pipeline — just tracing **one** syndrome row by hand through every
zone, to prove the architecture and the `source_record_id` scheme work before
scaling up. Uses `stable_record_hash` from the starter's `models.py`, the same
helper the real pipeline should use.

In [18]:
import hashlib
from quantum_lake_student.models import stable_record_hash

# ---- BRONZE ----
bronze_object = SYNDROMES_KEY
archive_member = "d-3_pfr-0.000010_nb-10M.csv"
record_locator = "row=1"  # 0-indexed data row after the header, within this member
bronze_object_sha256 = hashlib.sha256(_zip_cache[SYNDROMES_KEY]).hexdigest()

print("BRONZE")
print(" object:", bronze_object)
print(" member:", archive_member)
print(" locator:", record_locator)
print(" object sha256:", bronze_object_sha256)
print(" raw row:", row)  # from section 2a


BRONZE
 object: bronze/source=qec_syndromes/syndromes_dataset.zip
 member: d-3_pfr-0.000010_nb-10M.csv
 locator: row=1
 object sha256: bdfce36a71f04295ac78fb372d9c2e381801c05e3be119f919750ef59026d072
 raw row: ['1', '((1, 1, 1, 1), (1, 1, 1, 1), (1, 1, 1, 0), (1, 0, 0, 0))', '1']


In [19]:
# ---- SILVER ----
# silver/qec_syndromes/syndrome_observation.parquet -- one row per aggregate CSV row (silver-tables.md #1)
source_record_id = stable_record_hash({
    "source_name": "qec_syndromes",
    "bronze_object": bronze_object,
    "archive_member": archive_member,
    "record_locator": record_locator,
})

silver_row = {
    "source_record_id": source_record_id,
    "experiment_id": "qec_syndromes:d3:pfr=0.000010",   # stable id for this fault-rate group
    "physical_fault_rate": 0.000010,
    "syndrome_bits": bytes(flat16),
    "round_count": 4,
    "check_count": 4,
    "logical_error_label": bool(int(row[0])),
    "quantity": int(row[2]),
}
print("SILVER (silver/qec_syndromes/syndrome_observation.parquet)")
for k, v in silver_row.items():
    print(f"  {k}: {v}")


SILVER (silver/qec_syndromes/syndrome_observation.parquet)
  source_record_id: 1a9c24c147732fb006c4c805972bdd0385505b0b7d73562159f186f4aa11791e
  experiment_id: qec_syndromes:d3:pfr=0.000010
  physical_fault_rate: 1e-05
  syndrome_bits: b'\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00'
  round_count: 4
  check_count: 4
  logical_error_label: True
  quantity: 1


`source_record_id` is deterministic — same bytes in, same 64-hex-char hash out,
so a second run over unchanged Bronze reproduces it exactly (the repeatability
requirement in `silver-tables.md`). This same value also becomes a row in
`results/part1/source_trace.parquet`.

In [20]:
# ---- GOLD ----
# Student-designed PostgreSQL model. Sketching the row shapes for THIS record
# in two Gold tables (not real SQL yet -- that's next session's work).
gold_experiment_row = {
    "experiment_id": silver_row["experiment_id"],
    "source": "qec_syndromes",
    "code_distance": 3,
    "physical_fault_rate": silver_row["physical_fault_rate"],
}
gold_syndrome_observation_row = {
    "syndrome_observation_id": silver_row["source_record_id"],  # natural key also holds (see §5); reuse Silver's id as PK anyway
    "experiment_id": silver_row["experiment_id"],                # FK -> gold_experiment_row
    "syndrome_bits": silver_row["syndrome_bits"],
    "logical_error_label": silver_row["logical_error_label"],
    "quantity": silver_row["quantity"],
}
print("GOLD.experiment (one row this record belongs to)")
for k, v in gold_experiment_row.items():
    print(f"  {k}: {v}")
print("\nGOLD.syndrome_observation (this record)")
for k, v in gold_syndrome_observation_row.items():
    print(f"  {k}: {v}")


GOLD.experiment (one row this record belongs to)
  experiment_id: qec_syndromes:d3:pfr=0.000010
  source: qec_syndromes
  code_distance: 3
  physical_fault_rate: 1e-05

GOLD.syndrome_observation (this record)
  syndrome_observation_id: 1a9c24c147732fb006c4c805972bdd0385505b0b7d73562159f186f4aa11791e
  experiment_id: qec_syndromes:d3:pfr=0.000010
  syndrome_bits: b'\x00\x00\x01\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00'
  logical_error_label: True
  quantity: 1


In [21]:
# ---- ML ----
# ml/... table for Part II Task A: 16 syndrome_bits -> logical_error_label, weighted by sample_weight.
ml_row = {
    "example_id": stable_record_hash({"from": "gold.syndrome_observation",
                                       "id": gold_syndrome_observation_row["syndrome_observation_id"]}),
    "syndrome_bits": list(flat16),          # the ONLY model input at inference
    "logical_error_label": gold_syndrome_observation_row["logical_error_label"],
    "sample_weight": gold_syndrome_observation_row["quantity"],
    "physical_fault_rate": gold_experiment_row["physical_fault_rate"],  # kept for partitioning, NOT a feature
}
print("ML (ml/syndrome_examples...parquet)")
for k, v in ml_row.items():
    print(f"  {k}: {v}")

print()
print("Trace check: ml.example_id -> gold.syndrome_observation_id -> silver.source_record_id -> bronze row")
print(" ", ml_row["example_id"], "is derived from")
print(" ", gold_syndrome_observation_row["syndrome_observation_id"], "which equals")
print(" ", silver_row["source_record_id"], "which resolves to")
print(f"  bronze object={bronze_object}, member={archive_member}, locator={record_locator}")


ML (ml/syndrome_examples...parquet)
  example_id: 5dc37b48727d1f519aaed4d8f8944c20193b1e86d1362d7d2f9bbb6bad4b30e8
  syndrome_bits: [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  logical_error_label: True
  sample_weight: 1
  physical_fault_rate: 1e-05

Trace check: ml.example_id -> gold.syndrome_observation_id -> silver.source_record_id -> bronze row
  5dc37b48727d1f519aaed4d8f8944c20193b1e86d1362d7d2f9bbb6bad4b30e8 is derived from
  1a9c24c147732fb006c4c805972bdd0385505b0b7d73562159f186f4aa11791e which equals
  1a9c24c147732fb006c4c805972bdd0385505b0b7d73562159f186f4aa11791e which resolves to
  bronze object=bronze/source=qec_syndromes/syndromes_dataset.zip, member=d-3_pfr-0.000010_nb-10M.csv, locator=row=1


Run this cell yourself — `stable_record_hash` is deterministic, so you should
get exactly this same `example_id` if your Bronze bytes are unchanged. The
important part is the *chain*:
`ml.example_id → gold.syndrome_observation_id (== silver.source_record_id) → bronze (object, member, locator)`.
That's what "traceable" (rubric, `submission-checklist.md`) means concretely.

## Next session

- Do the same one-record trace for one **Google** shot (assemble across its 6
  companion files) — the rubric requires demonstrating one full syndrome trace
  *and* one full Google trace, not just one of the two.
- Turn §3/§4's profiling loops into real `stages/prepare_data.py` /
  `stages/register_sources.py` code that writes actual Silver Parquet files
  instead of just printing.
- Decide the Gold detector-storage strategy (one row per fired detector vs.
  packed/summary column) — open question in `decision_log.md`.
- Clear this notebook's outputs before committing, per `starter/notebooks/README.md`.
